# Load individual watch data runs and compile into one .parquet file

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
print(sorted(os.listdir("../data")))

['acc_watch_data.parquet', 'accumulated_npz', 'accumulated_weekend_data_2023_02_27.parquet', 'all_stabilties_settled_watch_data.parquet', 'minmax_watch_data.parquet', 'raw_text', 'raw_watch_data.parquet']


In [3]:
set_nums = ['00', '01', '02', '03', '04', '05', '06', '08', '08a', '08b', '08c', '08d', '09', '10', '10b', '11', '11b', '12', '13', '14'] # w/o set 7 bc only 67 features recorded

with open('../data/raw_text/column_names') as f:
    columns = [col.split()[1] for col in f.readlines()][1:]
    print(columns)

['m_over_q', 'fcv1_in', 'fcv1_i', 'gas_balzer_1', 'gas_balzer_2', 'gas_balzer_5', 'gas_balzer_6', 'gas_balzer_7', 'gas_name_1', 'gas_name_2', 'gas_name_5', 'gas_name_6', 'gas_name_7', 'inj_i', 'ext_i', 'mid_i', 'sext_i', 'inj_v', 'ext_v', 'mid_v', 'sext_v', 'inj_ps_v', 'ext_ps_v', 'mid_ps_v', 'sext_ps_v', 'inj_mbar', 'ext_mbar', 'bias_v', 'bias_i', 'k18_fw', 'k18_ref', 'g28_fw', 'glaser_1', 'batman_i', 'batman_field', 'x_ray_source', 'x_ray_exit', 'extraction_v', 'extraction_i', 'puller_v', 'puller_i', 'puller_raw_gap', 'bl_mig2_torr', 'robin_i', 'ht_oven_v', 'ht_oven_i', 'lt_oven_1_sp', 'lt_oven_2_sp', 'lt_oven_1_temp', 'lt_oven_2_temp', 'LHe_psi', 'LHe_level_percent', 'cryo_vac_torr', 'four_k_heater_power', 'four_k_cold_mass', 'four_k_cryo_e', 'four_k_cryo_w', 'four_k_cryo_ne', 'four_k_cryo_nw', 'four_k_heat_cond', 'four_k_i_feedthrough', 'four_k_heater_k', 'fifty_k_cond_bar', 'fifty_k_cond_bar_ne', 'fifty_k_cond_bar_nw', 'fifty_k_shield_bot', 'bottom_ln_vessel', 'seventy_k_cond_bar'

### Each run's data contains means ('m'), std ('s'), and time ('t') data in the following splits:
- 00:  unstable, not settled
- 01: unstable, settled
- 10: stable, unsettled
- 11: stable, settled

I will be using only stable, settled data. Going to get all the '11' data and make a dataframe using 'column_names' file. Adding a new column for which set # corresponds to the data.

In [4]:
import numpy as np
import pandas as pd

# Define the columns for mean and std
mean_cols = [col + "_mean" for col in columns]
std_cols = [col + "_std" for col in columns]

# Define the tags for the different statuses
status_mapping = {
    'm00': 'unstable_unsettled',
    'm01': 'unstable_settled',
    'm10': 'stable_unsettled',
    'm11': 'stable_settled'
}

set_data = []

for set in set_nums:
    path = f"../data/accumulated_npz/wd_{set}.npz"
    full_data = np.load(path)

    # Process data for each of the different stability/settling combinations
    for m_key, status in status_mapping.items():
        # Check if the dataset for this m_key is empty
        if m_key not in full_data or full_data[m_key].size == 0:
            print(f"No data found for {m_key} in set {set}. Skipping this category.")
            continue

        # Extract means, stds, and times for the given status
        means = pd.DataFrame(full_data[m_key], columns=mean_cols)
        std = pd.DataFrame(full_data[m_key.replace('m', 's')], columns=std_cols)
        times = pd.DataFrame(full_data[m_key.replace('m', 't')], columns=["unix_time"])

        # Ensure unix_time is in correct datetime format (if necessary)
        # Assuming unix_time is in seconds since epoch

        # Concatenate means, std, and times
        data = pd.concat([times, means, std], axis=1)

        # Add run_id and status columns
        data = data.assign(run_id=set, status=status)

        # Append the tagged data for this set
        set_data.append(data)

# Concatenate all sets into a single DataFrame
full_df = pd.concat(set_data, ignore_index=True)

# Sort the full dataframe by unix_time to preserve chronological order
full_df = full_df.sort_values(by='unix_time').reset_index(drop=True)

# The final full_df is now sorted by unix_time


No data found for m00 in set 04. Skipping this category.
No data found for m00 in set 08a. Skipping this category.
No data found for m00 in set 08b. Skipping this category.
No data found for m00 in set 08c. Skipping this category.
No data found for m00 in set 08d. Skipping this category.
No data found for m00 in set 09. Skipping this category.
No data found for m00 in set 10. Skipping this category.
No data found for m00 in set 11. Skipping this category.
No data found for m00 in set 11b. Skipping this category.
No data found for m00 in set 12. Skipping this category.
No data found for m00 in set 13. Skipping this category.


In [11]:
def counts_by_state(run_id):
    run_df = full_df[full_df['run_id'] == run_id]
    return run_df['status'].value_counts()

for run_id in full_df['run_id'].unique():
    counts = counts_by_state(run_id)
    print(f"{run_id} total: {sum(counts.values)}\n {counts}\n")

00 total: 3696
 status
stable_settled        2549
unstable_settled       798
stable_unsettled       306
unstable_unsettled      43
Name: count, dtype: int64

01 total: 819
 status
stable_settled        705
unstable_settled       75
stable_unsettled       34
unstable_unsettled      5
Name: count, dtype: int64

02 total: 1617
 status
stable_settled        1407
stable_unsettled       142
unstable_settled        59
unstable_unsettled       9
Name: count, dtype: int64

03 total: 410
 status
stable_settled        319
unstable_settled       51
stable_unsettled       36
unstable_unsettled      4
Name: count, dtype: int64

04 total: 20
 status
stable_settled      15
unstable_settled     4
stable_unsettled     1
Name: count, dtype: int64

05 total: 525
 status
stable_settled        428
unstable_settled       51
stable_unsettled       43
unstable_unsettled      3
Name: count, dtype: int64

06 total: 1239
 status
stable_settled        1092
unstable_settled       120
stable_unsettled        23
unst

### Save to parquet files
One dataframe with raw values, one with output current min-max scaled to be within the range 0-1 on a per set basis

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaled_df = full_df.copy()
scaled_df["fcv1_i_mean"] = scaled_df.groupby("run_id")["fcv1_i_mean"].transform(
    lambda x: scaler.fit_transform(x.values.reshape(-1, 1)).flatten()
)

full_df.to_parquet("all_stabilties_settled_watch_data.parquet")

# scaled_df.to_parquet("scaled_watch_data.parquet")

In [7]:
distributions = {}
for run_id in scaled_df.run_id.unique():
    set_df = scaled_df[scaled_df['run_id']==run_id]
    output = set_df["fcv1_i_mean"].to_numpy()
    distributions[run_id] = output


In [8]:
min_target = min(np.min(v) for v in distributions.values())
max_target = max(np.max(v) for v in distributions.values())

histograms = {run_id: np.histogram(targets, bins=50, range=(min_target, max_target), density=True)
              for run_id, targets in distributions.items()}
pdfs = {run_id: hist[0] for run_id, hist in histograms.items()}
pdfs

{'00': array([0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.01074694, 0.02364326, 0.04728653,
        0.08812489, 0.28371916, 0.36109711, 0.22353631, 0.25362773,
        0.3116612 , 0.2600759 , 0.21493876, 0.21923753, 0.18269794,
        0.23428325, 0.2944661 , 0.26867345, 0.32025875, 0.29016732,
        0.2428808 , 0.26867345, 0.25577712, 0.27297222, 0.38474038,
        0.40408486, 0.39118854, 0.3460514 , 0.40408486, 0.41268241,
        0.27942038, 0.27512161, 0.11821632, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ]),
 '01': array([0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.0581988 , 0.15519681, 0.0387992 ,
        0.17459641, 0.145497  , 0.16489661, 0.32979321, 0.18429621,
        0.30069381, 0.26189461, 0.